## How to read this notebook

The notebook follows the assignment's discovery order:

1. Load all six source files without changing their contents.
2. Produce structural profiles such as row counts, nulls, duplicates, ranges and category values.
3. Apply business-rule, format and referential-integrity checks.
4. Record each detected problem in a row-level Data Issues Log.
5. Save the profile summary and issue log for use when designing the ETL and ELT pipelines.

The comments inside each code cell explain the implementation details and the reason for each check.

# NammaMart Part 0 — Data profiling

This notebook profiles the six source files before any ETL or ELT code is written. It does not edit the input CSVs.

In [1]:
from pathlib import Path
import re
import pandas as pd

DATA_DIR = Path('include/data/nammamart')
OUTPUT_DIR = Path('assignment')
FILES = {
    'orders': 'orders.csv',
    'orders_increment': 'orders_2026-10-01_increment.csv',
    'payments': 'payments.csv',
    'customers': 'customers.csv',
    'products': 'products.csv',
    'stores': 'stores.csv',
}

data = {name: pd.read_csv(DATA_DIR / filename) for name, filename in FILES.items()}
print(f'Loaded {len(data)} files from {DATA_DIR.resolve()}')
for name, frame in data.items():
    print(f'{name:18} {len(frame):5,} rows x {len(frame.columns):2} columns')

Loaded 6 files from /Users/kumarsamanvaya/Documents/IITR - FDE Course/Data Pipepline Assignment/_Samy_IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/include/data/nammamart
orders             2,412 rows x 13 columns
orders_increment     160 rows x 13 columns
payments           2,228 rows x  7 columns
customers            405 rows x 10 columns
products             152 rows x 10 columns
stores                15 rows x  8 columns


In [2]:
# Basic profile: row counts, null counts, duplicate keys and ranges
KEYS = {
    'orders': 'order_id', 'orders_increment': 'order_id', 'payments': 'payment_id',
    'customers': 'customer_id', 'products': 'product_id', 'stores': 'store_id',
}
DATE_COLUMNS = {
    'orders': ['order_ts'], 'orders_increment': ['order_ts'], 'payments': ['payment_ts'],
    'customers': ['signup_date'], 'products': ['launch_date'], 'stores': ['open_date'],
}

profile_rows = []
for name, frame in data.items():
    key = KEYS[name]
    for column, count in frame.isna().sum().items():
        if count:
            profile_rows.append({'file': name, 'metric': 'nulls', 'column': column, 'value': int(count)})
    duplicate_count = int(frame[key].duplicated(keep=False).sum())
    profile_rows.append({'file': name, 'metric': 'duplicate_key_rows', 'column': key, 'value': duplicate_count})
    for column in frame.select_dtypes(include='number').columns:
        profile_rows.extend([
            {'file': name, 'metric': 'numeric_min', 'column': column, 'value': frame[column].min()},
            {'file': name, 'metric': 'numeric_max', 'column': column, 'value': frame[column].max()},
        ])
    for column in DATE_COLUMNS[name]:
        parsed = pd.to_datetime(frame[column], errors='coerce')
        profile_rows.extend([
            {'file': name, 'metric': 'date_min', 'column': column, 'value': parsed.min()},
            {'file': name, 'metric': 'date_max', 'column': column, 'value': parsed.max()},
        ])

profile = pd.DataFrame(profile_rows)
display(profile.head(40))
print('Categorical values:')
for name, frame in data.items():
    for column in frame.select_dtypes(include=['object', 'str']).columns:
        values = frame[column].dropna().astype(str).str.strip().unique()
        if len(values) <= 20:
            print(f'{name}.{column}: {sorted(values.tolist())}')

,file,metric,column,value
0,orders,nulls,order_ts,2
1,orders,nulls,customer_id,18
2,orders,nulls,delivery_minutes,164
3,orders,duplicate_key_rows,order_id,24
4,orders,numeric_min,quantity,-3
5,orders,numeric_max,quantity,5
6,orders,numeric_min,unit_price,10.0
7,orders,numeric_max,unit_price,883.0
8,orders,numeric_min,discount_pct,0
9,orders,numeric_max,discount_pct,250


Categorical values:
orders.store_id: ['S00', 'S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S17', 'S99']
orders.payment_mode: ['CARD', 'Card', 'Cash on Delivery', 'Cashh', 'UPI', 'Wallet', 'cod', 'upi']
orders.order_status: ['CANCELLED', 'DELIVERED', 'Pending??', 'RETURNED', 'delivered']
orders.channel: ['App', 'Web']
orders_increment.store_id: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15']
orders_increment.payment_mode: ['Card', 'Cash on Delivery', 'UPI', 'Wallet']
orders_increment.order_status: ['DELIVERED', 'RETURNED']
orders_increment.channel: ['App']
payments.payment_mode: ['CARD', 'Card', 'Cash on Delivery', 'Cashh', 'UPI', 'Wallet', 'cod', 'upi']
payments.payment_status: ['FAILED', 'REFUNDED', 'SUCCESS']
customers.city: ['Bengaluru']
customers.area: ['BTM Layout', 'Banashankari', 'Basaveshwaranagar', 'HSR Layout', 'Hebbal', 'Indiranagar', 'JP Nagar', 'Jayanagar',

In [3]:
# Build a row-level Data Issues Log
issues = []

def add_issues(file_name, mask, frame, row_key, column, problem, dimension, rule, severity='Critical'):
    mask = pd.Series(mask, index=frame.index).fillna(False).astype(bool)
    for index in frame.index[mask]:
        issues.append({
            'file': file_name,
            'row_key': frame.loc[index, row_key],
            'column': column,
            'problem': problem,
            'quality_dimension': dimension,
            'proposed_rule': rule,
            'severity': severity,
        })

orders = data['orders'].copy()
orders['order_ts_parsed'] = pd.to_datetime(orders['order_ts'], errors='coerce')
valid_status = {'DELIVERED', 'CANCELLED', 'RETURNED'}
valid_modes = {'UPI', 'Card', 'Cash on Delivery', 'Wallet'}
add_issues('orders.csv', orders['customer_id'].isna(), orders, 'order_id', 'customer_id', 'Missing customer_id', 'Completeness', 'customer_id must be present')
add_issues('orders.csv', orders['order_id'].duplicated(keep=False), orders, 'order_id', 'order_id', 'Duplicate order_id', 'Uniqueness', 'order_id must be unique')
add_issues('orders.csv', ~orders['order_status'].isin(valid_status), orders, 'order_id', 'order_status', 'Invalid order status', 'Validity', 'status must be DELIVERED, CANCELLED or RETURNED')
add_issues('orders.csv', ~orders['payment_mode'].isin(valid_modes), orders, 'order_id', 'payment_mode', 'Invalid payment mode', 'Validity', 'payment mode must be one of the allowed values')
add_issues('orders.csv', orders['quantity'] <= 0, orders, 'order_id', 'quantity', 'Quantity is not positive', 'Accuracy', 'quantity must be greater than zero')
expected_amount = (orders['quantity'] * orders['unit_price'] * (1 - orders['discount_pct'] / 100)).round(2)
add_issues('orders.csv', (orders['order_amount'] - expected_amount).abs() > 0.01, orders, 'order_id', 'order_amount', 'Amount does not match quantity, price and discount', 'Accuracy', 'order_amount must equal the pricing formula')
add_issues('orders.csv', (orders['order_status'] == 'DELIVERED') & orders['delivery_minutes'].isna(), orders, 'order_id', 'delivery_minutes', 'Delivered order has no delivery time', 'Completeness', 'DELIVERED orders require delivery_minutes')
add_issues('orders.csv', orders['delivery_minutes'].fillna(0) > 120, orders, 'order_id', 'delivery_minutes', 'Delivery time exceeds 120 minutes', 'Accuracy', 'delivery_minutes should be at most 120')
add_issues('orders.csv', orders['order_ts_parsed'] > pd.Timestamp.now(), orders, 'order_id', 'order_ts', 'Future-dated order', 'Timeliness', 'order_ts must not be in the future')

payments = data['payments'].copy()
add_issues('payments.csv', payments['payment_id'].duplicated(keep=False), payments, 'payment_id', 'payment_id', 'Duplicate payment_id', 'Uniqueness', 'payment_id must be unique')
add_issues('payments.csv', payments['gateway_ref'].isna(), payments, 'payment_id', 'gateway_ref', 'Missing gateway reference', 'Completeness', 'gateway_ref is required for payment auditability', 'Warning')
add_issues('payments.csv', payments['amount_paid'] <= 0, payments, 'payment_id', 'amount_paid', 'Payment amount is not positive', 'Accuracy', 'amount_paid must be greater than zero')

customers = data['customers'].copy()
add_issues('customers.csv', customers['customer_id'].duplicated(keep=False), customers, 'customer_id', 'customer_id', 'Duplicate customer_id', 'Uniqueness', 'customer_id must be unique')
add_issues('customers.csv', customers['city'].isna(), customers, 'customer_id', 'city', 'Missing city', 'Completeness', 'city should be present', 'Warning')
add_issues('customers.csv', ~customers['email'].fillna('').str.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$'), customers, 'customer_id', 'email', 'Invalid email format', 'Validity', 'email should have a valid email format', 'Warning')
add_issues('customers.csv', ~customers['phone'].fillna('').astype(str).str.fullmatch(r'\d{10}'), customers, 'customer_id', 'phone', 'Phone is not a 10-digit number', 'Validity', 'phone should contain exactly 10 digits', 'Warning')
add_issues('customers.csv', ~customers['pincode'].fillna('').astype(str).str.fullmatch(r'\d{6}'), customers, 'customer_id', 'pincode', 'Pincode is not six digits', 'Validity', 'pincode should contain exactly 6 digits', 'Warning')

products = data['products'].copy()
add_issues('products.csv', products['product_id'].duplicated(keep=False), products, 'product_id', 'product_id', 'Duplicate product_id', 'Uniqueness', 'product_id must be unique')
add_issues('products.csv', products['category'].isna(), products, 'product_id', 'category', 'Missing category', 'Completeness', 'category is required for category reporting')
add_issues('products.csv', products['mrp'] <= 0, products, 'product_id', 'mrp', 'MRP is not positive', 'Accuracy', 'mrp must be greater than zero')
add_issues('products.csv', products['cost_price'] > products['mrp'], products, 'product_id', 'cost_price', 'Cost price exceeds MRP', 'Accuracy', 'cost_price must not exceed mrp')

stores = data['stores'].copy()
add_issues('stores.csv', stores['store_id'].duplicated(keep=False), stores, 'store_id', 'store_id', 'Duplicate store_id', 'Uniqueness', 'store_id must be unique')
add_issues('stores.csv', stores['zone'].isna(), stores, 'store_id', 'zone', 'Missing zone', 'Completeness', 'zone is required for zone reporting')
add_issues('stores.csv', ~stores['pincode'].fillna('').astype(str).str.fullmatch(r'\d{6}'), stores, 'store_id', 'pincode', 'Pincode is not six digits', 'Validity', 'pincode should contain exactly 6 digits', 'Warning')
add_issues('stores.csv', stores['capacity_orders_per_day'] <= 0, stores, 'store_id', 'capacity_orders_per_day', 'Store capacity is not positive', 'Accuracy', 'capacity must be greater than zero')

# Referential integrity
add_issues('orders.csv', ~orders['customer_id'].isin(set(customers['customer_id'])), orders, 'order_id', 'customer_id', 'Unknown customer_id', 'Consistency', 'orders.customer_id must exist in customers')
add_issues('orders.csv', ~orders['product_id'].isin(set(products['product_id'])), orders, 'order_id', 'product_id', 'Unknown product_id', 'Consistency', 'orders.product_id must exist in products')
add_issues('orders.csv', ~orders['store_id'].isin(set(stores['store_id'])), orders, 'order_id', 'store_id', 'Unknown store_id', 'Consistency', 'orders.store_id must exist in stores')
add_issues('payments.csv', ~payments['order_id'].isin(set(orders['order_id'])), payments, 'payment_id', 'order_id', 'Unknown order_id', 'Consistency', 'payments.order_id must exist in orders')

# Payment amount reconciliation for successful payments
payment_check = payments.merge(orders[['order_id', 'order_amount']], on='order_id', how='left')
mismatch = (payment_check['payment_status'] == 'SUCCESS') & payment_check['order_amount'].notna() & ((payment_check['amount_paid'] - payment_check['order_amount']).abs() > 0.01)
add_issues('payments.csv', mismatch, payment_check, 'payment_id', 'amount_paid', 'Successful payment does not equal order amount', 'Consistency', 'SUCCESS payment must equal order_amount')

# Profile the next-day feed with the same critical order rules.
increment = data['orders_increment'].copy()
increment['order_ts_parsed'] = pd.to_datetime(increment['order_ts'], errors='coerce')
increment_expected_amount = (increment['quantity'] * increment['unit_price'] * (1 - increment['discount_pct'] / 100)).round(2)
add_issues('orders_2026-10-01_increment.csv', increment['customer_id'].isna(), increment, 'order_id', 'customer_id', 'Missing customer_id', 'Completeness', 'customer_id must be present')
add_issues('orders_2026-10-01_increment.csv', ~increment['order_status'].isin(valid_status), increment, 'order_id', 'order_status', 'Invalid order status', 'Validity', 'status must be DELIVERED, CANCELLED or RETURNED')
add_issues('orders_2026-10-01_increment.csv', (increment['order_amount'] - increment_expected_amount).abs() > 0.01, increment, 'order_id', 'order_amount', 'Amount does not match pricing formula', 'Accuracy', 'order_amount must equal the pricing formula')
add_issues('orders_2026-10-01_increment.csv', increment['order_ts_parsed'] > pd.Timestamp.now(), increment, 'order_id', 'order_ts', 'Future-dated order', 'Timeliness', 'order_ts must not be in the future')

issues_log = pd.DataFrame(issues)
print(f'Found {len(issues_log):,} issue rows')
display(issues_log.head(30))

Found 530 issue rows


,file,row_key,column,problem,quality_dimension,proposed_rule,severity
0,orders.csv,ORD002255,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
1,orders.csv,ORD000701,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
2,orders.csv,ORD000997,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
3,orders.csv,ORD000580,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
4,orders.csv,ORD002019,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
5,orders.csv,ORD002262,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
6,orders.csv,ORD002149,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
7,orders.csv,ORD002367,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
8,orders.csv,ORD001332,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
9,orders.csv,ORD001918,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical


In [4]:
# Summaries required for the assignment
if issues_log.empty:
    issue_summary = pd.DataFrame(columns=['file', 'quality_dimension', 'severity', 'issue_rows'])
else:
    issue_summary = (issues_log.groupby(['file', 'quality_dimension', 'severity'])
                    .size().reset_index(name='issue_rows')
                    .sort_values(['file', 'quality_dimension']))
display(issue_summary)
print('Issues by file:')
display(issues_log.groupby('file').size().rename('issue_rows').to_frame())

OUTPUT_DIR.mkdir(exist_ok=True)
issues_log.to_csv(OUTPUT_DIR / 'data_issues_log.csv', index=False)
profile.to_csv(OUTPUT_DIR / 'data_profile_summary.csv', index=False)
print(f"Wrote {OUTPUT_DIR / 'data_issues_log.csv'}")
print(f"Wrote {OUTPUT_DIR / 'data_profile_summary.csv'}")

,file,quality_dimension,severity,issue_rows
0,customers.csv,Completeness,Warning,6
1,customers.csv,Uniqueness,Critical,10
2,customers.csv,Validity,Warning,22
3,orders.csv,Accuracy,Critical,34
4,orders.csv,Completeness,Critical,26
5,orders.csv,Consistency,Critical,32
6,orders.csv,Timeliness,Critical,5
7,orders.csv,Uniqueness,Critical,24
8,orders.csv,Validity,Critical,15
9,payments.csv,Accuracy,Critical,10


Issues by file:


,issue_rows
file,
customers.csv,38
orders.csv,136
payments.csv,341
products.csv,13
stores.csv,2


Wrote assignment/data_issues_log.csv
Wrote assignment/data_profile_summary.csv


## Questions to confirm with the client

1. Should a returned order reduce the original September revenue after the October correction arrives, or should finance preserve both snapshots?
2. Should warning-level customer PII format issues be corrected, masked and loaded, or quarantined?
3. When multiple payments exist for one order, which payment is authoritative for reconciliation?